**Figure 1: Overhead Accumulation and Optimization.** Experiments are conducted on
the deterministic 64-call coding workload (2 repeats). (a) walks the optimization
chain of full-mode AgentTX; (b) compares current execution modes, with the bare
lower bound marked as a red dotted reference. Results suggest that repeated per-call
userspace setup, not isolation itself, accounts for the main overhead: the
persistent try worker cuts full-mode latency by ~61%, and the remaining gap to
`shared_checkpoint` is dominated by read tracing.


In [ ]:
# ipython -c "%run plot.ipynb"
# Shared USENIX plotting convention (FAST/OSDI camera-ready).
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import style
import pandas as pd
import numpy as np
from pathlib import Path

STANDARD_WIDTH = 17.8            # USENIX two-column text width, in cm
SINGLE_COL_WIDTH = STANDARD_WIDTH / 2
DOUBLE_COL_WIDTH = STANDARD_WIDTH

def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update(plt.rcParamsDefault)
matplotlib.rcParams['text.usetex'] = False
style.use('bmh')
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.linewidth'] = 0.5
plt.rcParams['hatch.linewidth'] = 0.5
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['font.family'] = 'Nimbus Roman'
pd.options.display.max_columns = None

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'

history = pd.read_csv(RESULTS / 'motivation_optimization_history.csv')
runtime = pd.read_csv(RESULTS / 'motivation_runtime_comparison.csv')
history_full = history[history['metric'] == 'full_ms_per_step'].copy()

fig = plt.figure(dpi=300, figsize=(cm_to_inch(DOUBLE_COL_WIDTH), cm_to_inch(3.2)))
plt.rcParams['axes.grid.axis'] = 'y'
colors = ['#2b2d42', '#8d99ae', '#4ecdc4', '#1a535c', '#ef233c']
bar_width = 0.34

# (a) Optimization chain: paired before/after bars.
ax0 = plt.subplot(1, 2, 1)
x = np.arange(len(history_full))
before = history_full['before'].astype(float).to_numpy()
after = history_full['after'].astype(float).to_numpy()
bars_before = ax0.bar(x - bar_width / 2, before, width=bar_width, hatch='///', color=colors[1], linewidth=0.5, label='before')
bars_after = ax0.bar(x + bar_width / 2, after, width=bar_width, color=colors[4], linewidth=0.5, label='after')
ax0.set_xticks(x, labels=['W/D\ntrace', 'READ/NEG\neffects', 'script\nreuse', 'defer\nGC', 'direct\nscript', 'persistent\nworker'], fontsize=6)
ax0.set_ylabel('Full AgentTX latency (ms/step)', fontsize=8)
ax0.tick_params(bottom=False, top=False, left=False, right=False)
ax0.tick_params(axis='y', labelsize=8)
ax0.set_title('(a) Optimization chain', fontsize=8)
worker_idx = len(history_full) - 1
ax0.annotate(f"-{(1 - after[worker_idx] / before[worker_idx]):.0%}",
             (worker_idx + bar_width / 2, after[worker_idx]),
             textcoords='offset points', xytext=(0, 4), ha='center', fontsize=6, color=colors[4])

# (b) Current baselines, with the bare lower bound as a red dotted reference line.
ax1 = plt.subplot(1, 2, 2)
runtime_order = ['bare', 'per_call_try', 'shared_try', 'shared_checkpoint', 'agenttx_without_read_tracing', 'agenttx_full']
runtime_names = ['bare', 'per-call\ntry', 'shared\ntry', 'shared\ncheckpoint', 'AgentTX\nno-trace', 'AgentTX\nfull']
runtime = runtime.set_index('mode').loc[runtime_order].reset_index()
values = runtime['per_step_mean_ms'].astype(float).to_numpy()
bar_colors = [colors[0], colors[1], colors[1], colors[2], colors[3], colors[4]]
bars = ax1.bar(np.arange(len(values)), values, width=0.62, color=bar_colors, linewidth=0.5, hatch=['', '///', '///', '', '', ''])
ax1.axhline(values[0], color=colors[4], linewidth=0.7, linestyle=':', zorder=1)
for i, v in enumerate(values[1:], start=1):
    ax1.annotate(f"{v / values[0]:.1f}x", (i, v), textcoords='offset points', xytext=(0, 3), ha='center', fontsize=6)
ax1.set_xticks(np.arange(len(values)), labels=runtime_names, fontsize=6)
ax1.set_ylabel('Latency (ms/step)', fontsize=8)
ax1.tick_params(bottom=False, top=False, left=False, right=False)
ax1.tick_params(axis='y', labelsize=8)
ax1.set_title('(b) Current baselines', fontsize=8)

for ax in fig.axes:
    for axis in ['top', 'bottom', 'left', 'right']:
        ax.spines[axis].set_linewidth(0.5)

fig.legend([bars_before[0], bars_after[0]], ['before', 'after'], loc='upper center', bbox_to_anchor=(0.5, 1.12), ncol=2, frameon=False, columnspacing=1.0, handlelength=1.2, fontsize=8)
plt.tight_layout(pad=0.4, rect=[0.045, 0.0, 0.99, 0.88])
plt.savefig(FIGDIR / 'FIG-Motivation-Optimization.pdf', bbox_inches='tight', pad_inches=0)
plt.savefig(FIGDIR / 'FIG-Motivation-Optimization.png', dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()

chain_start, chain_end = float(before[0]), float(after[-1])
print(f"optimization chain: {chain_start:.1f} -> {chain_end:.1f} ms/step ({(1 - chain_end / chain_start):.1%})")
print(f"full vs bare: {values[-1] / values[0]:.2f}x; full vs checkpoint floor: {values[-1] / values[3]:.2f}x")
